# 05 — Clustering me K-Means
**Dataset:** Credit Card Fraud Detection — të dhëna të parapërpunuara nga `02_preprocessing.ipynb`

Qëllimi: Zbulojmë struktura natyrore në të dhëna **pa përdorur labels** (unsupervised learning). Krahasojmë clusterat e gjetur me klasat reale për të vlerësuar sa mirë K-Means ndan transaksionet mashtruese nga ato legjitime.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

RANDOM_STATE = 42

In [ ]:
X_all         = np.load('../data/processed/X_all.npy')
y_all         = np.load('../data/processed/y_all.npy')
feature_names = np.load('../data/processed/feature_names.npy', allow_pickle=True)

print(f'X_all shape    : {X_all.shape}')
print(f'y_all shape    : {y_all.shape}')
print(f'Features ({len(feature_names)}): {list(feature_names)}')
print(f'\nKlasa 0 (legjitime) : {(y_all == 0).sum():,}')
print(f'Klasa 1 (mashtruese): {(y_all == 1).sum():,}')
print(f'\nLabels NUK përdoren gjatë clustering — unsupervised!')

## 1. Zgjedhja e K Optimal — Elbow Method
Testojmë K nga 2 deri 10 dhe shikojmë kur inertia (shuma e distancave brenda clusterit) ndalon së uluri ndjeshëm — "bërryli" i grafikut tregon K optimal.

In [ ]:
K_range  = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_all)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_all, km.labels_, sample_size=10000, random_state=RANDOM_STATE)
    silhouettes.append(sil)
    print(f'K={k}  |  Inertia: {km.inertia_:,.0f}  |  Silhouette: {sil:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_title('Elbow Method — Inertia sipas K', fontweight='bold')
axes[0].set_xlabel('Numri i Clusterave (K)')
axes[0].set_ylabel('Inertia')
axes[0].axvline(x=2, color='tomato', linestyle='--', linewidth=1.5, label='K optimal')
axes[0].legend()

axes[1].plot(list(K_range), silhouettes, 'gs-', linewidth=2, markersize=8)
axes[1].set_title('Silhouette Score sipas K', fontweight='bold')
axes[1].set_xlabel('Numri i Clusterave (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].axvline(x=2, color='tomato', linestyle='--', linewidth=1.5, label='K optimal')
axes[1].legend()

plt.suptitle('Zgjedhja e K Optimal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/elbow_method.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nK=2 zgjidhet: dataset ka 2 klasa natyrore (legjitime / mashtruese)')

## 2. Trajnimi i K-Means (K=2)
Trajnojmë modelin final me K=2. Clusterat gjenden **pa labels** — vetëm bazuar në distancat mes pikave.

In [ ]:
kmeans = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10)
kmeans.fit(X_all)
labels_pred = kmeans.labels_

print("=" * 50)
print("REZULTATET E K-MEANS (K=2)")
print("=" * 50)
print(f"Inertia finale    : {kmeans.inertia_:,.2f}")
print(f"Silhouette Score  : {silhouette_score(X_all, labels_pred, sample_size=10000, random_state=RANDOM_STATE):.4f}")
print(f"Iteracione        : {kmeans.n_iter_}")
print(f"\nShpërndarja e clusterave:")
unique, counts = np.unique(labels_pred, return_counts=True)
for cl, cnt in zip(unique, counts):
    print(f"  Cluster {cl}: {cnt:>7,}  ({cnt / len(labels_pred) * 100:.2f}%)")

## 3. Vizualizimi me PCA (2D)
Reduktojmë dimensionalitetin nga 30 në 2 komponente kryesore (PCA) për të vizualizuar clusterat në hapësirë 2D.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_all)

print(f"Varianca e shpjeguar:")
print(f"  PC1: {pca.explained_variance_ratio_[0]*100:.2f}%")
print(f"  PC2: {pca.explained_variance_ratio_[1]*100:.2f}%")
print(f"  Total: {pca.explained_variance_ratio_.sum()*100:.2f}%")

In [ ]:
idx0 = np.where(labels_pred == 0)[0]
idx1 = np.where(labels_pred == 1)[0]

sample0 = np.random.choice(idx0, size=min(5000, len(idx0)), replace=False)
sample1 = np.random.choice(idx1, size=min(5000, len(idx1)), replace=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(X_pca[sample0, 0], X_pca[sample0, 1],
                c='steelblue', alpha=0.3, s=5, label=f'Cluster 0 ({len(idx0):,})')
axes[0].scatter(X_pca[sample1, 0], X_pca[sample1, 1],
                c='tomato', alpha=0.3, s=5, label=f'Cluster 1 ({len(idx1):,})')
centers_pca = pca.transform(kmeans.cluster_centers_)
axes[0].scatter(centers_pca[:, 0], centers_pca[:, 1],
                c='black', marker='X', s=200, zorder=5, label='Centroide')
axes[0].set_title('K-Means Clusterat (K=2)', fontweight='bold')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend(markerscale=3)

fraud_idx  = np.where(y_all == 1)[0]
legit_idx  = np.where(y_all == 0)[0]
s_legit    = np.random.choice(legit_idx, size=5000, replace=False)

axes[1].scatter(X_pca[s_legit, 0], X_pca[s_legit, 1],
                c='steelblue', alpha=0.3, s=5, label=f'Legjitime ({len(legit_idx):,})')
axes[1].scatter(X_pca[fraud_idx, 0], X_pca[fraud_idx, 1],
                c='tomato', alpha=0.6, s=15, label=f'Mashtruese ({len(fraud_idx):,})')
axes[1].set_title('Klasat Reale (për krahasim)', fontweight='bold')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend(markerscale=3)

plt.suptitle('Vizualizimi PCA 2D — Clusterat vs Klasat Reale', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()